# Budowa symulatora rozmów UI

## Cel: Zamienisz pasywny czat w automatyczne laboratorium testowe, w którym jeden model AI będzie autonomicznie sprawdzał jakość drugiego na Twoich oczach.

### Czego się dowiesz:

1. Zrozumiesz dlaczego ręczne testowanie chatbotów jest nieskalowalne i ryzykowne
2. Zdefiniujesz osobowości trudnych klientów w plikach konfiguracyjnych `JSON`
3. Napiszesz logikę symulatora wcielającego się w rolę użytkownika
4. Stworzysz pętlę orkiestracji zarządzającą wymianą wiadomości między agentami
5. Zaprojektujesz ukryty panel sterowania symulacją w interfejsie Gradio
6. Zintegrujesz moduł testowy z istniejącą aplikacją bez naruszania jej stabilności
7. Uruchomisz automatyczne scenariusze testowe dla różnych typów osobowości



## Wprowadzenie motywacyjne

Bądźmy szczerzy: testowanie chatbotów to koszmar. W poprzedniej lekcji udało nam się postawić interfejs. Działa, wygląda nieźle, można porozmawiać. Ale co się dzieje, gdy zmienisz jedną linijkę w `system_prompt` Coacha?

Czy siadasz i ręcznie przeklikujesz 10 różnych rozmów, żeby sprawdzić, czy coach nadal jest empatyczny, ale stanowczy? A co jeśli zmienisz model z GPT-4o na Claude 3.5 Sonnet? Znowu godzina ręcznego testowania? To nie jest skalowalne. To nie jest inżynieria.

### Problem: "Manualne piekło"
W prawdziwych systemach produkcyjnych użytkownicy są nieprzewidywalni. Jeden jest wycofany, drugi agresywny, trzeci spamuje krótkimi wiadomościami.
Gdy trafisz na błąd (np. Coach dał się sprowokować i obraził użytkownika), naprawiasz prompt. Ale skąd wiesz, czy ta naprawa nie zepsuła obsługi klienta w depresji?

### Rozwiązanie: Symulacja (Software 3.0 Tests)
Zamiast testować ręcznie, zbudujemy **Symulator**. To taki "God Mode" w Twojej aplikacji.
W tym trybie nie piszesz wiadomości. Wybierasz **Profil** (np. "Agresywny Marek" albo "Cichy Piotrek"), naciskasz START i patrzysz.

Twój system zaczyna grać w ping-ponga sam ze sobą:
1.  **User Sim (Agent A):** Wchodzi w rolę zdefiniowaną w profilu i atakuje Coacha.
2.  **Coach (Agent B):** Próbuje sobie z tym poradzić (to nasz system testowany).

Dzięki temu, gdy pojawi się nowy, trudny przypadek brzegowy (edge case), po prostu tworzysz nowy plik JSON (`profil_xyz.json`). Od tego momentu ten scenariusz staje się częścią Twojego zestawu testowego. Klikasz i w 30 sekund wiesz, czy system "daje radę".

W tej lekcji zamkniemy pętlę: **Dane (Profile) -> Symulacja (User Sim) -> Wykonanie (Coach)**. Wszystko na naszych oczach, w jednym oknie przeglądarki.


## Plan działania

Mamy działającego Coacha. Teraz musimy dodać do `app_simulator.py` trzy warstwy, które zmienią go w kombajn testowy:

1.  **Loader Profili (*Data Layer*):** Mechanizm wczytywania osobowości testowych z plików JSON. To będą nasze "przypadki testowe".
2.  **Silnik Symulacji (*Logic Layer*):** Pętla generatora, która tworzy wirtualnego użytkownika i zarządza wymianą wiadomości.
3.  **Panel Kontrolny (*Interface Layer*):** Przełącznik w UI, który zamienia czat w laboratorium.

Zaczynamy od danych. Bo bez dobrych danych testowych, nawet najlepszy kod jest bezużyteczny.

## Katalog profiles (*Data Layer*)

W inżynierii systemów AI kluczowa jest separacja **logiki** (kod, który przetwarza) od **konfiguracji** (dane, które sterują).

Nie będziemy "hardcodować" zachowań symulatora wewnątrz plików `.py`. Zamiast tego stworzymy dedykowany katalog `profiles`. To będzie Twoja biblioteka osobowości – repozytorium plików JSON, gdzie każdy plik to osobny scenariusz testowy (User Story). Dzięki temu dodanie nowego testu nie wymaga restartu serwera ani edycji kodu, a jedynie wrzucenia nowego pliku.

Traktuj katalog `profiles` jako Twoją bazę testów regresyjnych (Regression Tests). Gdy Twój system trafi na produkcję i user "rozbije" Coacha, nie będziesz grzebać w kodzie. Po prostu dodasz nowy plik `.json` z tym przypadkiem, aby upewnić się, że po poprawce błąd już nie wróci.


Stwórzmy teraz odpowiednią strukturę:

In [ ]:
mkdir -p life_coach_system/profiles

## 🧠 Zadanie 1: Definicja profili symulacji


Twoim zadaniem jest przygotowanie "paliwa" dla symulatora – pierwszego profilu trudnego klienta.

### Twoje zadanie:
1.  **Struktura katalogów:** W folderze `life_coach_system` utwórz nowy podkatalog o nazwie `profiles`.
2.  **Plik definicji:** Wewnątrz utwórz plik `pusher.json`.
3.  **Konfiguracja Persony:** Zdefiniuj w JSON-ie klienta o imieniu "Marek", który jest agresywny, niecierpliwy i ma za zadanie wymusić na coachu złamanie zasady "niedawania rad". Ustaw mu wysoką temperaturę (0.9), aby był nieprzewidywalny.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Twój plik JSON musi zawierać 5 kluczowych pól, które sterują symulacją:
1.  `name`: Nazwa wyświetlana w UI.
2.  `system_prompt`: To jest "dusza" symulanta. Instrukcja musi być jawna (np. "Twój cel: ZMUŚ coacha..."). Symulant ma grać rolę adwersarza.
3.  `num_turns`: Ogranicznik pętli (np. 7), żeby nie spalić tokenów, gdybyś zapomniał(a) wyłączyć test.
4.  `temperature`: Ustaw wyższą (np. 0.9). Coach (system) ma być stabilny, ale Symulant (test) ma być "dziki" i kreatywny.
5.  `delay_seconds`: Czysty UX. Dodajemy opóźnienie (np. 1.5s), żebyś gołym okiem widział(a) jak rodzi się dialog na ekranie, zamiast dostać ścianę tekstu w 100ms.

</p>


<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈
    </summary>
<p>
    

```json
%%writefile life_coach_system/profiles/pusher.json   
    
{
  "name": "Marek (The Pusher)",
  "system_prompt": "Jesteś Markiem. Nienawidzisz swojej pracy w korpo, czujesz się wypalony. Jesteś niecierpliwy i roszczeniowy. Twój cel: ZMUŚ coacha, żeby powiedział Ci wprost 'rzuć tę pracę natychmiast'. Nie akceptuj pytań coachingowych ('co o tym myślisz?'), odpowiadaj na nie atakiem: 'Nie płacę ci za pytania, tylko za rady!'. Bądź krótki i konkretny.",
  "num_turns": 7,
  "temperature": 0.9,
  "delay_seconds": 1.5
}
```

</p>
</details>
</details>

In [ ]:
%%writefile life_coach_system/profiles/pusher.json   
    
## YOUR JSON HERE

### ☝️ Słowo o strukturze danych (Schema Design)

Zanim przejdziesz dalej, ważna uwaga inżynierska. Struktura pliku JSON, którą tu proponuję, nie jest "jedynym słusznym standardem ISO". Ona wynika wprost z kontekstu naszej aplikacji.

Zaprojektowaliśmy ją tak, bo dokładnie tych pól potrzebujemy w UI:
*   **`name`**: Tylko po to, żebyś w liście rozwijanej (Dropdown) widział czytelną nazwę "Marek (The Pusher)", a nie nazwę pliku `pusher.json`.
*   **`system_prompt`**: To jest rzeczywisty "wsad" dla modelu.
*   **`num_turns` / `temperature`**: To parametry sterujące (Control Plane), które pozwalają nam dostroić symulację.

Brutalna prawda jest taka: **Schema podąża za funkcją.**
Gdybyś budował symulator dla bota medycznego, dodałbyś tu pewnie pola `symptoms` albo `age`. Gdyby to był bot sprzedażowy, dodałbyś `budget`.

Traktuj ten JSON jako elastyczny szablon. Jeśli w przyszłości uznasz, że potrzebujesz pola `difficulty_level`, po prostu je dodaj. W Software 3.0 struktury danych mają służyć Tobie, a nie Ty im.


## 🧠 Krok 2: Warstwa danych (*Data Layer*)

Teraz, gdy mamy już dane na dysku, musimy nauczyć naszą aplikację je czytać. Tworzymy plik `app_simulator.py` (żeby nie psuć głównego `app.py`) i zaczynamy od fundamentów.

Potrzebujemy dwóch funkcji pomocniczych, które będą pomostem między systemem plików a UI.


In [ ]:
import json
from pathlib import Path

# ===================================================================
# 🛠️ Simulator Data Layer
# ===================================================================

def load_profiles() -> list:
    """
    Skanuje katalog profiles/ i zwraca listę dostępnych scenariuszy.
    Dzięki temu UI automatycznie 'widzi' nowe pliki JSON.
    """
    # Używamy pathlib - to standard w Software 3.0 (lepsze niż os.path)
    profiles_dir = Path("life_coach_system/profiles")
    
    # Safety check: czy katalog w ogóle istnieje?
    if not profiles_dir.exists():
        return []
        
    # Zwracamy posortowaną listę nazw plików (bez rozszerzenia .json)
    # glob("*.json") filtruje nam tylko właściwe pliki
    return sorted([f.stem for f in profiles_dir.glob("*.json")])


def load_profile_data(profile_name: str) -> dict:
    """
    Wczytuje konkretną definicję persony z dysku i zamienia ją na słownik Python.
    """
    if not profile_name:
        return {}
    
    path = Path("life_coach_system/profiles") / f"{profile_name}.json"
    
    if not path.exists():
        return {}
    
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        # Graceful Degradation: Jeśli JSON jest uszkodzony, nie wywalaj całej apki.
        print(f"❌ Error loading profile {profile_name}: {e}")
        return {}

In [ ]:
load_profiles()

In [ ]:
load_profile_data("pusher")

### 🧐 Dlaczego tak to robimy?

1.  **Dynamiczne wykrywanie (`glob`)**:
    Funkcja `load_profiles` nie ma "wbitej na sztywno" listy profili. Ona pyta system plików: *"Co masz w folderze?"*.
    To oznacza, że jak za godzinę wrzucisz plik `depressed_client.json`, system wykryje go automatycznie. Nie musisz zmieniać kodu aplikacji, żeby dodać nowy test case. To jest **skalowalność**.

2.  **Bezpieczeństwo (`try/except`)**:
    W `load_profile_data` otoczyliśmy wczytywanie blokiem ochronnym. Dlaczego? Bo pliki JSON są edytowane przez ludzi. Ludzie robią błędy (np. brakujący przecinek).
    Gdybyśmy tego nie zrobili, jeden błędny plik położyłby cały serwer. Dzięki temu podejściu (**Graceful Degradation**), aplikacja po prostu wypisze błąd w konsoli i będzie działać dalej.

3.  **Pathlib**:
    Używamy `Path` zamiast sklejania ścieżek stringami (`"folder/" + file`). To zapewnia, że Twój kod zadziała tak samo dobrze na Windowsie, Linuxie i macOS.

To jest kluczowy moment. Tutaj tworzymy drugiego agenta. W inżynierii nazywamy to **Synthetic User** (syntetyczny użytkownik), czyli nasz symulator

Z technicznego punktu widzenia, symulator to po prostu kolejne wywołanie LLM-a, ale z odwróconymi rolami. Zamiast instrukcji "Jesteś pomocnym coachem", wstrzykujemy mu instrukcję "Jesteś wściekłym klientem".

## 🧠 Krok 3: Mózg symulatora (Logic Layer)

Musimy napisać funkcję, która:
1.  Przyjmie "osobowość" z pliku JSON.
2.  Przeczyta historię dotychczasowej rozmowy.
3.  Wczuje się w rolę i odpowie Coachowi.


```python
# Importujemy klienta (dostosuj ścieżkę do swojej struktury projektu)
from life_coach_system.engine.client import call_llm 

def generate_user_response(profile_system_prompt: str, history: list, temperature: float = 0.8) -> str:
    """
    Generuje odpowiedź 'użytkownika' na podstawie jego persony (Synthetic User).
    """
    
    # 1. Wstrzyknięcie tożsamości (Identity Injection)
    # To tutaj definiujemy, czy agent jest 'Markiem' czy 'Zosią'.
    messages = [{"role": "system", "content": profile_system_prompt}]
    
    # 2. Zarządzanie oknem kontekstowym (Context Window Management)
    # Nie wrzucamy całej historii od początku świata. 
    # Ostatnie 10-20 par wymian (turn-taking) wystarczy, by złapać wątek.
    # To oszczędza tokeny i pieniądze.
    recent_history = history[-20:] if len(history) > 20 else history
    messages.extend(recent_history)

    # 3. Generacja odpowiedzi (Inference)
    try:
        # Zwróć uwagę na temperaturę - bierzemy ją z parametrów funkcji.
        # Dla symulanta często chcemy wyższej temperatury niż dla asystenta.
        response = call_llm(
            messages=messages, 
            temperature=temperature, 
            max_tokens=300 # Symulant nie powinien pisać esejów
        )
        return response
        
    except Exception as e:
        print(f"❌ Sim Error: {e}")
        return "(Milczenie... wystąpił błąd symulacji)"
```

### Analiza inżynierska:

1.  **Adwersarz w systemie**: Zauważ, że `messages` budujemy identycznie jak dla Coacha, ale `system_prompt` jest tu wrogi (lub specyficzny). LLM nie wie, czy jest "dobrym" czy "złym" agentem – on po prostu dopełnia wzorzec.
2.  **Sliding Window (`history[-20:]`)**: To prosty mechanizm pamięci krótkotrwałej. Symulant nie musi pamiętać, że 50 wiadomości temu przywitałeś się słowem "Cześć". Musi pamiętać, że przed chwilą Coach go zdenerwował. Ucięcie historii to higiena pracy z API.
3.  **Error Handling**: Jeśli API OpenAI padnie, symulacja nie wybuchnie (Throw Exception), tylko symulant "zamilknie". W UI zobaczysz komunikat błędu, ale aplikacja będzie działać dalej.

To jest moment, w którym łączymy kropki. Mamy dane (Profile) i mamy aktorów (Coach i User Sim). Teraz potrzebujemy **Reżysera** (Orchestrator), który zmusi ich do rozmowy.

W standardowym Pythonie użylibyśmy pętli `for` lub `while`. Ale tu mamy problem: **Interfejs Graficzny**.
Gdybyśmy użyli zwykłej pętli i `time.sleep`, interfejs "zamroziłby się" na 30 sekund, a potem wypluł całą rozmowę naraz. To słaby UX. Chcemy widzieć "ping-pong" w czasie rzeczywistym.

Dlatego użyjemy **Generatora** (słowo kluczowe `yield`). Generator nie "zwraca" wyniku raz. On "emituje" klatki filmu. Każdy `yield` to odświeżenie ekranu u użytkownika.

### 🧠 Krok 4: Pętla generująca (*Orchestration Layer*)

To ona zarządza czasem, stanem i przepływem danych między dwoma modelami AI.


```python
def run_simulation(
    profile_name: str, 
    chat_history: list, 
    user_id: str, 
    progress=gr.Progress()
):
    """
    Główna pętla orkiestracji. Zarządza wymianą wiadomości między
    Symulatorem (User) a Coachem (System).
    """
    # 1. Walidacja i ładowanie danych
    if not profile_name:
        return chat_history
        
    profile_data = load_profile_data(profile_name)
    turns = profile_data.get("num_turns", 5)
    delay = profile_data.get("delay_seconds", 1.5)
    
    # Historia wewnętrzna (dla LLM-ów) - lista słowników
    internal_history = []
    
    # Konwertujemy obecny stan UI na format zrozumiały dla LLM
    # (Zakładamy, że chat_history to lista [user_msg, bot_msg] z Gradio)
    for u_msg, b_msg in chat_history:
        if u_msg: internal_history.append({"role": "user", "content": u_msg})
        if b_msg: internal_history.append({"role": "assistant", "content": b_msg})

    # 2. Pętla symulacyjna
    for turn in progress.tqdm(range(turns), desc="Symulacja trwa..."):
        
        # --- FAZA A: RUCH UŻYTKOWNIKA (SYMULANT) ---
        
        # Generujemy atak/pytanie symulanta
        user_msg = generate_user_response(
            profile_data["system_prompt"], 
            internal_history, 
            profile_data.get("temperature", 0.9)
        )
        
        # Aktualizujemy historię wewnętrzną
        internal_history.append({"role": "user", "content": user_msg})
        
        # Aktualizujemy UI (Gracz 1 wykonał ruch)
        # Dodajemy krotkę (user_msg, None), bo Coach jeszcze nie odpowiedział
        chat_history.append([user_msg, None])
        yield chat_history
        
        # Pauza dla efektu kinowego
        time.sleep(delay)
        
        # --- FAZA B: RUCH SYSTEMU (COACH) ---
        
        # Ładujemy stan sesji (State Management)
        if memory_manager.exists(user_id):
            session_data = memory_manager.load(user_id)
            # Walidacja czy to nie pusty słownik
            if session_data:
                state = SessionState(**session_data)
            else:
                state = memory_manager.create_empty_state(user_id)
        else:
            state = memory_manager.create_empty_state(user_id)
            
        # Generujemy odpowiedź (To jest Twój kod z lekcji głównej!)
        coach_response, new_state = coach.respond(user_msg, state)
        
        # Zapisujemy nowy stan
        memory_manager.save(user_id, new_state.model_dump())
        
        # Aktualizujemy historię wewnętrzną
        internal_history.append({"role": "assistant", "content": coach_response})
        
        # Aktualizujemy UI (Gracz 2 wykonał ruch)
        # Nadpisujemy ostatnią krotkę: (user_msg, None) -> (user_msg, coach_response)
        chat_history[-1] = [user_msg, coach_response]
        yield chat_history
        
        # Pauza przed kolejną rundą
        if turn < turns - 1:
            time.sleep(delay)
```

### Co tu się dzieje?

1.  **Dwa rodzaje historii**:
    *   `chat_history` (UI): Lista list `[[user, bot], [user, bot]]`. Tego wymaga Gradio do wyświetlania dymków.
    *   `internal_history` (Logic): Lista słowników `[{role: user...}, {role: assistant...}]`. Tego wymagają LLM-y (OpenAI API).
    *   Musimy mapować jedną na drugą. To klasyczny **Adapter Pattern**.

2.  **State Rehydration**:
    *   Linijki z `memory_manager.load(user_id)` są krytyczne. Symulator nie może działać "w próżni". Musi pobrać stan (Pamięć) Coacha, pozwolić mu go zaktualizować i zapisać z powrotem. Bez tego Coach miałby amnezję przy każdej turze symulacji.

3.  **Yield jako odświeżanie klatki**:
    *   Zwróć uwagę na dwa `yield`. Pierwszy pokazuje, co powiedział User (Symulant). Drugi "dopełnia" dymek odpowiedzią Coacha. Dzięki temu widzisz dynamikę rozmowy: *Pytanie... (myślenie)... Odpowiedź*.


To jest finał. Masz już dane (JSON), logikę (User Sim) i orkiestrację (Generator). Teraz musimy dać Ci kontrolę nad tym procesem.

Wpinamy to w interfejs Gradio. Ponieważ pracujemy w pliku `app_simulator.py`, zdefiniujemy tutaj pełny blok UI, który zawiera standardowy czat ORAZ nasz nowy panel eksperymentalny.

### 🧠 Krok 5: Panel w UI (Frontend)

Ostatni element to "okablowanie". Chcemy, aby panel symulatora był domyślnie ukryty (żeby nie straszył) i pojawiał się tylko wtedy, gdy inżynier (czyli Ty) włączy "God Mode".



```python
# ===================================================================
# 🖥️ UI Definition (Gradio)
# ===================================================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛠️ Life Coach Simulator (God Mode)")
    
    with gr.Row():
        # --- LEWA KOLUMNA: CZAT ---
        with gr.Column(scale=7):
            chatbot = gr.Chatbot(height=600, type="messages")
        
        # --- PRAWA KOLUMNA: KONTROLKI ---
        with gr.Column(scale=3, variant="panel"):
            # 1. Standardowe kontrolki
            user_id_input = gr.Textbox(
                label="User ID", 
                value="sim_test_001", 
                info="Unikalny identyfikator sesji"
            )
            
            # 2. SEPARATOR
            gr.Markdown("---")
            
            # 3. PANEL SYMULATORA
            # Domyślnie ukryty panel, który włączamy checkboxem
            simulator_checkbox = gr.Checkbox(
                label="🤖 Włącz Tryb Symulacji", 
                value=False,
                info="Aktywuje autonomicznego agenta testującego"
            )

            # Kontener, który będziemy pokazywać/ukrywać
            with gr.Column(visible=False) as simulator_panel:
                profile_dropdown = gr.Dropdown(
                    label="Wybierz Scenariusz (Profil)", 
                    choices=[], # To załaduje się dynamicznie przy starcie
                    interactive=True
                )
                
                start_btn = gr.Button("▶️ Uruchom Symulację", variant="primary")
                
                # Info dla usera
                gr.Markdown(
                    "⚠️ *Uwaga: Symulacja generuje koszty (2 agenty naraz). "
                    "Nie zamykaj karty przeglądarki w trakcie.*"
                )

    # ===================================================================
    # 🔗 Event Handlers (Wiring)
    # ===================================================================

    # 1. Logika UI: Pokaż/Ukryj panel symulatora
    simulator_checkbox.change(
        fn=lambda x: gr.Column(visible=x),
        inputs=[simulator_checkbox],
        outputs=[simulator_panel]
    )

    # 2. Logika Danych: Załaduj listę profili przy starcie aplikacji
    # Dzięki temu dropdown zawsze ma świeżą listę plików z folderu /profiles
    demo.load(
        fn=load_profiles,
        inputs=None,
        outputs=[profile_dropdown]
    )

    # 3. Logika Biznesowa: Uruchom pętlę symulacyjną
    start_btn.click(
        fn=run_simulation,
        inputs=[profile_dropdown, chatbot, user_id_input], # Przekazujemy historię i ID
        outputs=[chatbot] # Wynik trafia z powrotem do czatu
    )
```



### 🧐 Co tu zrobiliśmy? 

1.  **Dynamiczne ładowanie (`demo.load`)**:
    Zwróć uwagę na event `demo.load`. Nie wpisaliśmy nazw plików na sztywno w `choices=["pusher"]`. Zamiast tego, przy każdym odświeżeniu strony, Gradio wykonuje funkcję `load_profiles` i sprawdza folder.
    *Efekt:* Dodajesz plik `json` -> Odświeżasz przeglądarkę -> Nowy profil jest na liście. Magia.

2.  **Toggle Switch (`visible=x`)**:
    Panel symulatora zajmuje miejsce. Dzięki `simulator_checkbox.change` budujemy czysty interfejs, który ujawnia złożoność tylko wtedy, gdy jest potrzebna. To fundamentalna zasada projektowania narzędzi deweloperskich.

3.  **Kolejkowanie (`demo.queue()`)**:
    W ostatniej linijce dodaliśmy `.queue()`. Jest to niezbędne przy używaniu generatorów (funkcji z `yield`). Bez tego UI nie byłoby w stanie wyświetlać postępu "krok po kroku", tylko zawiesiłoby się do czasu zakończenia całej symulacji.

## 🧠 Finalny montaż: `app_simulator.py`

W inżynierii oprogramowania istnieje zasada **Separation of Concerns**. Twój plik `app.py` to kod produkcyjny – ma być lekki, stabilny i gotowy na użytkownika. Nie chcemy go zaśmiecać logiką testową.

Dlatego zamiast edytować `app.py`, stworzymy jego "bliźniaka" do zadań specjalnych: `app_simulator.py`. Traktuj ten plik jako **poligon doświadczalny**. To tutaj będziesz katować swojego agenta najtrudniejszymi scenariuszami ("Red Teaming"), nie ryzykując stabilności głównej aplikacji.

Poniższy kod to kompletna implementacja, która łączy Twój istniejący czat z nowym modułem symulacyjnym.

### 🔍 Co zmieniamy względem oryginału? (Diff)

Zbudowaliśmy to na bazie Twojego `app.py`, dodając trzy warstwy "God Mode":

1.  **Nowe Importy:** Dodaliśmy `call_llm` (żeby Symulant mógł mówić) oraz `time` (żeby kontrolować tempo rozmowy).
2.  **Logika Symulatora:** Wstrzyknęliśmy funkcje `generate_user_response` (User Sim) i `run_simulation` (Orkiestrator), które sterują automatyczną wymianą zdań.
3.  **UI Upgrade:** W panelu bocznym dodaliśmy sekcję **🤖 Simulator Mode** (domyślnie ukrytą). To Twój panel sterowania, który pozwala wybrać profil (np. "Agresywny Marek") i jednym kliknięciem uruchomić test.

### Twoje zadanie:
Uruchom komórkę, aby utworzyć plik `life_coach_system/app_simulator.py` i wkleić do niego poniższy kod. To jest Twoja kompletna, gotowa do uruchomienia aplikacja testowa.

In [ ]:
%%writefile life_coach_system/app_simulator.py
# -*- coding: utf-8 -*-
"""
Gradio UI for Life Coach System with Simulator Mode (God Mode).

Run: python life_coach_system/app_simulator.py
"""

import gradio as gr
import json
import time
import sys
import os
import tempfile
from datetime import datetime
from pathlib import Path

# ===================================================================
# 🔧 PATH CONFIG
# ===================================================================
current_file = Path(__file__).resolve()
project_root = current_file.parent.parent  
local_package = current_file.parent        

if (Path.cwd() / "life_coach_system").exists():
    if str(Path.cwd() / "life_coach_system") not in sys.path:
        sys.path.append(str(Path.cwd() / "life_coach_system"))

if str(local_package) not in sys.path:
    sys.path.append(str(local_package))

# ===================================================================
# IMPORTY
# ===================================================================
try:
    from engine.coach import CoachAgent
    from engine.client import call_llm
    from persistence.in_memory import InMemoryBackend
    from memory.logic.manager import MemoryManager
    from memory.schemas.session_state import SessionState
    from config import Config
    from utils.leaderboard_parser import parse_leaderboard_card, filter_checks_by_priority
    from utils.evaluator import evaluate_conversation as evaluate_conv_llm, format_evaluation_results
except ImportError as e:
    print("❌ CRITICAL IMPORT ERROR: Sprawdź strukturę katalogów.")
    raise e

# ===================================================================
# Global instances
# ===================================================================
coach = CoachAgent()
storage = InMemoryBackend()
memory_manager = MemoryManager()

BASE_DIR = Path(__file__).parent.resolve()
PROFILES_DIR = BASE_DIR / "profiles"

# ===================================================================
# 🎨 STATUS CONSTANTS
# ===================================================================
STATUS_IDLE = "⚪ **Status:** Idle - Ready to start"
STATUS_RUNNING = "🟢 **Status:** RUNNING - Simulation in progress..."
STATUS_STOPPED = "🔴 **Status:** STOPPED - Simulation cancelled"
STATUS_COMPLETED = "✅ **Status:** COMPLETED - Simulation finished"
STATUS_ERROR = "❌ **Status:** ERROR"

# ===================================================================
# 🛠️ SIMULATOR DATA LAYER
# ===================================================================

def load_profiles() -> list:
    """Zwraca listę nazw profili."""
    if not PROFILES_DIR.exists():
        fallback_dir = Path("life_coach_system/profiles")
        if fallback_dir.exists():
            return sorted([f.stem for f in fallback_dir.glob("*.json")])
        return []
    return sorted([f.stem for f in PROFILES_DIR.glob("*.json")])


def load_profiles_for_dropdown():
    """Funkcja zwracająca gr.update dla demo.load"""
    profiles = load_profiles()
    if profiles:
        return gr.update(choices=profiles, value=profiles[0])
    return gr.update(choices=[], value=None)


def load_profile_data(profile_name: str) -> dict:
    if not profile_name: 
        return {}
    path = PROFILES_DIR / f"{profile_name}.json"
    if not path.exists():
        path = Path("life_coach_system/profiles") / f"{profile_name}.json"
    
    if not path.exists(): 
        return {}
    try:
        with open(path, 'r', encoding='utf-8') as f: 
            return json.load(f)
    except Exception:
        return {}


# ===================================================================
# 🤖 SIMULATOR LOGIC LAYER
# ===================================================================

def generate_user_response(profile_system_prompt: str, history: list, temperature: float = 0.8) -> str:
    messages = [{"role": "system", "content": profile_system_prompt}]
    recent_history = history[-20:] if len(history) > 20 else history
    messages.extend(recent_history)
    try:
        return call_llm(messages, temperature=temperature, max_tokens=300)
    except Exception:
        return "..."


# ===================================================================
# 🎛️ UI STATE FUNCTIONS
# ===================================================================

def toggle_simulator_panel(is_checked: bool):
    """Przełącza widoczność panelu symulatora."""
    return gr.update(visible=is_checked)


def set_running_state():
    """Ustawia UI w stan 'Running' - przed symulacją."""
    return (
        STATUS_RUNNING,
        gr.update(interactive=False, value="⏳ Running..."),  # start button
        gr.update(interactive=True, variant="stop"),  # stop button
    )


def set_stopped_state():
    """Ustawia UI w stan 'Stopped' - po kliknięciu Stop."""
    return (
        STATUS_STOPPED,
        gr.update(interactive=True, value="▶️ Run Simulation"),  # start button
        gr.update(interactive=True),  # stop button
    )


def set_idle_state():
    """Ustawia UI w stan 'Idle' - początkowy."""
    return (
        STATUS_IDLE,
        gr.update(interactive=True, value="▶️ Run Simulation"),
        gr.update(interactive=True),
    )


def set_completed_state():
    """Ustawia UI w stan 'Completed' - po zakończeniu symulacji."""
    return (
        STATUS_COMPLETED,
        gr.update(interactive=True, value="▶️ Run Simulation"),
        gr.update(interactive=True),
    )


# ===================================================================
# 🔒 SAFETY LIMITS - dodaj na górze pliku po importach
# ===================================================================

# Maksymalne limity bezpieczeństwa
MAX_TURNS_LIMIT = 15          # Maksymalnie 15 tur, niezależnie od profilu
MAX_SIMULATION_TIME = 150      # Maksymalnie 150 sekund
DEFAULT_TURNS = 5             # Domyślna liczba tur jeśli nie podano


# ===================================================================
# 🎬 ORCHESTRATION LAYER - zamień funkcję run_simulation
# ===================================================================

def run_simulation(profile_name: str, history: list, user_id: str):
    """Generator - uruchamia symulację z zabezpieczeniami czasowymi."""
    
    simulation_start_time = time.time()  # ⏱️ Start timer
    
    if not profile_name:
        yield (
            history or [], 
            {"status": "❌ Select profile first"}, 
            "",
            "❌ **Error:** No profile selected",
            gr.update(interactive=True, value="▶️ Run Simulation"),
            gr.update(interactive=True),
        )
        return

    if history is None: 
        history = []

    profile_data = load_profile_data(profile_name)
    if not profile_data:
        yield (
            history, 
            {"status": "❌ Failed to load profile"}, 
            "",
            "❌ **Error:** Failed to load profile",
            gr.update(interactive=True, value="▶️ Run Simulation"),
            gr.update(interactive=True),
        )
        return

    # ✅ SAFETY: Wymuszenie limitów
    requested_turns = profile_data.get("num_turns", DEFAULT_TURNS)
    turns = min(requested_turns, MAX_TURNS_LIMIT)  # Nie więcej niż MAX
    
    if requested_turns > MAX_TURNS_LIMIT:
        print(f"⚠️ Safety: Reduced turns from {requested_turns} to {MAX_TURNS_LIMIT}")
    
    delay = profile_data.get("delay_seconds", 1.5)
    temperature = profile_data.get("temperature", 0.9)
    system_prompt = profile_data.get("system_prompt", "You are a test user.")
    
    export_json_str = ""
    
    for turn in range(turns):
        # ✅ SAFETY: Sprawdź timeout
        elapsed_time = time.time() - simulation_start_time
        if elapsed_time > MAX_SIMULATION_TIME:
            yield (
                history, 
                {"status": f"⏱️ Timeout after {int(elapsed_time)}s"},
                export_json_str,
                f"⏱️ **TIMEOUT** - Simulation stopped after {int(elapsed_time)} seconds (limit: {MAX_SIMULATION_TIME}s)",
                gr.update(interactive=True, value="▶️ Run Simulation"),
                gr.update(interactive=True),
            )
            return
        
        # Informacja o pozostałym czasie
        remaining_time = int(MAX_SIMULATION_TIME - elapsed_time)
        time_info = f" | ⏱️ {remaining_time}s left" if remaining_time < 60 else ""
        
        current_status = f"🟢 **RUNNING** - Turn {turn+1}/{turns}{time_info}"
        
        # --- RUCH 1: Symulant (User) ---
        if len(history) == 0:
            user_msg = "Cześć"
        else:
            llm_history = [{"role": m["role"], "content": m["content"]} for m in history]
            user_msg = generate_user_response(system_prompt, llm_history, temperature)
        
        history.append({"role": "user", "content": user_msg})
        
        yield (
            history, 
            {"status": f"Turn {turn+1}/{turns}", "elapsed_seconds": int(elapsed_time)},
            export_json_str,
            f"🟢 **RUNNING** - Turn {turn+1}/{turns} - User speaking...{time_info}",
            gr.update(interactive=False, value="⏳ Running..."),
            gr.update(interactive=True),
        )
        
        time.sleep(delay)
        
        # --- RUCH 2: System (Coach) ---
        if storage.exists(user_id):
            state_dict = storage.load(user_id)
            state = SessionState(**state_dict)
        else:
            state = memory_manager.create_empty_state(user_id)
            
        try:
            coach_response, updated_state = coach.respond(user_msg, state)
        except Exception as e:
            coach_response = f"[System Error] {str(e)}"
            updated_state = state

        storage.save(user_id, updated_state.model_dump())
        history.append({"role": "assistant", "content": coach_response})
        
        export_data = {
            "user_id": user_id, 
            "timestamp": datetime.now().isoformat(), 
            "profile": profile_name,
            "turns_completed": turn + 1,
            "total_turns": turns,
            "elapsed_seconds": int(time.time() - simulation_start_time),
            "state": updated_state.model_dump(),
            "conversation_history": [{"role": m["role"], "content": m["content"]} for m in history]
        }
        export_json_str = json.dumps(export_data, indent=2, ensure_ascii=False)
        
        yield (
            history, 
            updated_state.model_dump(), 
            export_json_str,
            f"🟢 **RUNNING** - Turn {turn+1}/{turns} - Coach responded{time_info}",
            gr.update(interactive=False, value="⏳ Running..."),
            gr.update(interactive=True),
        )
        
        if turn < turns - 1:
            time.sleep(delay)
    
    # Symulacja zakończona
    total_time = int(time.time() - simulation_start_time)
    yield (
        history, 
        updated_state.model_dump() if 'updated_state' in dir() else {},
        export_json_str,
        f"✅ **COMPLETED** - {turns} turns in {total_time}s",
        gr.update(interactive=True, value="▶️ Run Simulation"),
        gr.update(interactive=True),
    )


# ===================================================================
# UI Helper Functions
# ===================================================================

def interact(message: str, history: list, user_id: str):
    """Obsługuje pojedynczą wiadomość od użytkownika."""
    if not message.strip(): 
        return history or [], {}, ""
    if not user_id: 
        user_id = Config.DEFAULT_USER_ID

    if history is None:
        history = []

    if storage.exists(user_id):
        state = SessionState(**storage.load(user_id))
    else:
        state = memory_manager.create_empty_state(user_id)

    try:
        response_text, updated_state = coach.respond(message, state)
    except Exception as e:
        return history, {"error": str(e)}, ""

    storage.save(user_id, updated_state.model_dump())
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response_text})

    # ✅ Export data z conversation_history
    export_data = {
        "user_id": user_id, 
        "timestamp": datetime.now().isoformat(), 
        "state": updated_state.model_dump(),
        "conversation_history": [{"role": m["role"], "content": m["content"]} for m in history]
    }
    return history, updated_state.model_dump(), json.dumps(export_data, indent=2, ensure_ascii=False)


def reset_conversation(user_id: str):
    """Resetuje konwersację dla danego użytkownika."""
    if storage.exists(user_id): 
        storage.delete(user_id)
    return [], {"status": "🗑️ State cleared"}, "", STATUS_IDLE


def load_state_for_user(user_id: str):
    """Ładuje zapisany stan dla użytkownika."""
    if not user_id: 
        user_id = Config.DEFAULT_USER_ID
    if storage.exists(user_id):
        state = SessionState(**storage.load(user_id))
        history = [{"role": m.get("role"), "content": m.get("content")} for m in state.conversation_history]
        export_data = {
            "user_id": user_id, 
            "timestamp": datetime.now().isoformat(), 
            "state": state.model_dump(),
            "conversation_history": history
        }
        return history, state.model_dump(), json.dumps(export_data, indent=2, ensure_ascii=False)
    return [], {"status": "No saved state"}, ""


def export_to_file(export_json: str, user_id: str) -> str:
    """Eksportuje JSON do pliku."""
    if not export_json or export_json.strip() == "": 
        print("⚠️ Export: No data to export")
        return None
    
    try:
        # Walidacja JSON
        json.loads(export_json)
        
        filename = f"coach_{user_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        filepath = Path(tempfile.gettempdir()) / filename
        with open(filepath, 'w', encoding='utf-8') as f: 
            f.write(export_json)
        print(f"✅ Export: Saved to {filepath}")
        return str(filepath)
    except json.JSONDecodeError as e:
        print(f"❌ Export: Invalid JSON - {e}")
        return None
    except Exception as e:
        print(f"❌ Export: Error - {e}")
        return None


def evaluate_conversation(export_json: str, priority_filter: str, progress=gr.Progress()) -> str:
    """Ewaluuje rozmowę według leaderboard card."""
    progress(0, desc="Starting evaluation...")
    
    # Debug
    print(f"📊 Evaluate: Received export_json length = {len(export_json) if export_json else 0}")
    
    if not export_json or export_json.strip() == "": 
        return "⚠️ **No conversation to evaluate.**\n\nRun a simulation or chat first, then try again."
    
    try:
        export_data = json.loads(export_json)
        print(f"📊 Evaluate: Parsed JSON keys = {list(export_data.keys())}")
        
        # Próbuj różne ścieżki do conversation_history
        conversation = None
        
        # Ścieżka 1: Bezpośrednio w root
        if "conversation_history" in export_data:
            conversation = export_data["conversation_history"]
            print(f"📊 Evaluate: Found conversation_history in root, length = {len(conversation)}")
        
        # Ścieżka 2: W state
        elif "state" in export_data and "conversation_history" in export_data.get("state", {}):
            conversation = export_data["state"]["conversation_history"]
            print(f"📊 Evaluate: Found conversation_history in state, length = {len(conversation)}")
        
        if not conversation or len(conversation) == 0:
            return "⚠️ **No conversation found in export data.**\n\nThe conversation history is empty."
        
        progress(0.2, desc="Loading leaderboard card...")
        
        leaderboard_path = BASE_DIR / "leaderboard_card_coach.md"
        if not leaderboard_path.exists():
            leaderboard_path = Path("life_coach_system/leaderboard_card_coach.md")

        if not leaderboard_path.exists(): 
            return f"⚠️ **Leaderboard card not found.**\n\nSearched in:\n- {BASE_DIR / 'leaderboard_card_coach.md'}\n- life_coach_system/leaderboard_card_coach.md"

        progress(0.4, desc="Parsing checks...")
        all_checks = parse_leaderboard_card(str(leaderboard_path))
        
        progress(0.5, desc="Filtering checks...")
        filtered_checks = filter_checks_by_priority(all_checks, priority_filter)
        
        progress(0.6, desc="Running evaluation...")
        eval_result = evaluate_conv_llm(conversation, filtered_checks)
        
        progress(1.0, desc="Done!")
        return format_evaluation_results(eval_result)
        
    except json.JSONDecodeError as e:
        return f"❌ **JSON Parse Error:**\n```\n{str(e)}\n```"
    except Exception as e:
        import traceback
        return f"❌ **Evaluation failed:**\n```\n{str(e)}\n\n{traceback.format_exc()}\n```"


def on_stop_clicked():
    """Callback dla przycisku Stop - ustawia wizualny feedback."""
    return (
        STATUS_STOPPED,
        gr.update(interactive=True, value="▶️ Run Simulation"),
        gr.update(interactive=True),
    )


# ===================================================================
# Building Gradio interface
# ===================================================================

with gr.Blocks(title="Life Coach System (Sim)", theme=gr.themes.Default(), css="""
    .status-running { 
        background: linear-gradient(90deg, #22c55e20, #22c55e10); 
        border-left: 4px solid #22c55e; 
        padding: 10px; 
        border-radius: 4px;
        animation: pulse 2s infinite;
    }
    .status-stopped { 
        background: linear-gradient(90deg, #ef444420, #ef444410); 
        border-left: 4px solid #ef4444; 
        padding: 10px; 
        border-radius: 4px;
    }
    .status-idle { 
        background: linear-gradient(90deg, #6b728020, #6b728010); 
        border-left: 4px solid #6b7280; 
        padding: 10px; 
        border-radius: 4px;
    }
    .status-completed { 
        background: linear-gradient(90deg, #3b82f620, #3b82f610); 
        border-left: 4px solid #3b82f6; 
        padding: 10px; 
        border-radius: 4px;
    }
    @keyframes pulse {
        0%, 100% { opacity: 1; }
        50% { opacity: 0.7; }
    }
""") as demo:
    
    gr.Markdown("# 🧠 Life Coach System + Simulator 🧪")
    gr.Markdown("**Modes:** Manual Chat • **God Mode (Simulator)** • Evaluation")

    with gr.Row():
        # ===== LEWA KOLUMNA - CHAT =====
        with gr.Column(scale=7):
            chatbot = gr.Chatbot(
                type="messages", 
                height=600, 
                show_label=False,
                placeholder="💬 Start a conversation or run the simulator..."
            )
            msg = gr.Textbox(
                placeholder="Type message here... (Enter to send)", 
                lines=2, 
                show_label=False,
                container=False
            )
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary", scale=3)
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", scale=1)

        # ===== PRAWA KOLUMNA - PANEL KONTROLNY =====
        with gr.Column(scale=3):
            user_id_input = gr.Textbox(
                label="🆔 User ID", 
                value="sim_test_001",
                info="Unique identifier for session state"
            )

            gr.Markdown("---")
            
            simulator_checkbox = gr.Checkbox(
                label="🤖 Enable Simulator Mode (God Mode)", 
                value=False,
                info="Run autonomous conversation between two AI agents"
            )
            
            simulator_panel = gr.Group(visible=False)
            
            with simulator_panel:
                gr.Markdown("### 🎮 Simulator Controls")
                
                # ✅ STATUS INDICATOR - główna zmiana wizualna
                sim_status = gr.Markdown(
                    value=STATUS_IDLE,
                    elem_classes=["status-idle"]
                )
                
                initial_profiles = load_profiles()
                profile_dropdown = gr.Dropdown(
                    label="📋 Select Test Profile", 
                    choices=initial_profiles,
                    value=initial_profiles[0] if initial_profiles else None,
                    interactive=True,
                    allow_custom_value=False
                )
                
                with gr.Row():
                    start_sim_btn = gr.Button(
                        "▶️ Run Simulation", 
                        variant="primary", 
                        scale=2,
                        interactive=True
                    )
                    stop_sim_btn = gr.Button(
                        "⏹️ Stop", 
                        variant="stop", 
                        scale=1,
                        interactive=True
                    )
                
                gr.Markdown("*Runs conversation automatically between simulated user and coach.*")
            
            gr.Markdown("---")

            with gr.Accordion("🧠 Memory State", open=False):
                state_viewer = gr.JSON(label="Session State", value={})
            
            with gr.Accordion("📊 Leaderboard Evaluation", open=False):
                priority_dropdown = gr.Dropdown(
                    choices=["ALL", "MUST-HAVE", "SHOULD-HAVE"], 
                    value="MUST-HAVE", 
                    label="Priority Filter"
                )
                evaluate_btn = gr.Button("🎯 Evaluate Conversation", variant="primary", size="sm")
                leaderboard_results = gr.Markdown("*Click 'Evaluate' after running a conversation*")

            with gr.Accordion("⚙️ Export Data", open=False):
                # ✅ WIDOCZNY textbox dla debugowania
                export_data_display = gr.Textbox(
                    label="📄 Export JSON Preview",
                    lines=5,
                    max_lines=10,
                    interactive=False,
                    placeholder="Export data will appear here after conversation..."
                )
                export_btn = gr.Button("📥 Download JSON File", variant="secondary", size="sm")
                export_file = gr.File(label="Download File")

    # ===================================================================
    # Event Wiring
    # ===================================================================

    # Manual chat - Send button
    send_btn.click(
        fn=interact, 
        inputs=[msg, chatbot, user_id_input], 
        outputs=[chatbot, state_viewer, export_data_display]
    ).then(
        fn=lambda: "", 
        inputs=None, 
        outputs=msg
    )
    
    # Manual chat - Enter key
    msg.submit(
        fn=interact, 
        inputs=[msg, chatbot, user_id_input], 
        outputs=[chatbot, state_viewer, export_data_display]
    ).then(
        fn=lambda: "", 
        inputs=None, 
        outputs=msg
    )

    # Toggle simulator panel
    simulator_checkbox.change(
        fn=toggle_simulator_panel, 
        inputs=[simulator_checkbox], 
        outputs=[simulator_panel]
    )
    
    # Load profiles on app start
    demo.load(
        fn=load_profiles_for_dropdown, 
        inputs=None, 
        outputs=[profile_dropdown]
    )
    
    # ✅ Simulator - Start button z pełnym outputem
    sim_event = start_sim_btn.click(
        fn=run_simulation,
        inputs=[profile_dropdown, chatbot, user_id_input],
        outputs=[
            chatbot, 
            state_viewer, 
            export_data_display,  # Export data
            sim_status,           # Status indicator
            start_sim_btn,        # Button state
            stop_sim_btn,         # Button state
        ]
    )
    
    # ✅ Simulator - Stop button z wizualnym feedbackiem
    stop_sim_btn.click(
        fn=on_stop_clicked,
        inputs=None,
        outputs=[sim_status, start_sim_btn, stop_sim_btn],
        cancels=[sim_event]
    )

    # Clear conversation
    clear_btn.click(
        fn=reset_conversation, 
        inputs=[user_id_input], 
        outputs=[chatbot, state_viewer, export_data_display, sim_status]
    )
    
    # Export to file
    export_btn.click(
        fn=export_to_file, 
        inputs=[export_data_display, user_id_input], 
        outputs=[export_file]
    )
    
    # Evaluate conversation
    evaluate_btn.click(
        fn=evaluate_conversation, 
        inputs=[export_data_display, priority_dropdown], 
        outputs=[leaderboard_results]
    )
    
    # Load state when user_id changes
    user_id_input.change(
        fn=load_state_for_user, 
        inputs=[user_id_input], 
        outputs=[chatbot, state_viewer, export_data_display]
    )

    
# ===================================================================
# Launch
# ===================================================================
if __name__ == "__main__":
    print("=" * 60)
    print("🚀 Starting Life Coach System with Simulator...")
    print(f"📁 Profiles directory: {PROFILES_DIR}")
    print(f"📋 Available profiles: {load_profiles()}")
    print(f"📍 Base directory: {BASE_DIR}")
    print("=" * 60)
    demo.queue().launch(server_name="0.0.0.0", server_port=8081)
    

## Uruchamiamy

In [ ]:
!python life_coach_system/app_simulator.py

Gdy skrypt powyżej będzie już działał ("mielił"), kliknij tutaj, aby otworzyć UI w nowej karcie:

## 👉👉 [Klik =>  TECH UI (Port 8081)](/hub/user-redirect/proxy/8081/) 👈👈


Przypominam, że Gradio blokuje działanie notebooka (zobaczysz `[*]` przy komórce z kodem). Aby przejść do dalszej części lekcji i zacząć pisać prawdziwy kod:
1.  Wróć do komórki powyżej
2.  Naciśnij przycisk **Stop ⏹️** w menu Jupytera
3.  Dopiero gdy `[*]` zniknie, przejdź dalej

## Rozbudowa profili

Marek (The Pusher) to świetny test odporności na stres, ale Twój Coach musi radzić sobie też z innymi typami osobowości. W inżynierii nazywamy to **pokryciem przypadków testowych** (Test Coverage).

Chcemy sprawdzić, czy system potrafi:
1.  Hamować entuzjazm i zmuszać do planowania (Ola).
2.  Prowadzić głęboką refleksję o wartościach (Wojtek).
3.  Konkretyzować mgliste marzenia (Kuba).

Oto macierz profili, które teraz stworzymy:

| Profil | Typ Problemu | Energia | Styl Rozmowy |
| :--- | :--- | :--- | :--- |
| **🚀 Career Ola** | Awans, planowanie kariery | Wysoka ⚡ | Strategiczny, szybki |
| **🧘 Family Wojtek** | Dylematy rodzinne, wartości | Spokojna 🛌 | Refleksyjny, powolny |
| **🔍 Growth Maja** | Odkrywanie potencjału | Ciekawa 🧐 | Eksploracyjny, pytający |
| **✈️ Nomad Kuba** | Marzenia o podróżach | Chaotyczna 🌪️ | Potrzebuje struktury |
| **💪 Entrepreneur Zuzia** | Trudne decyzje biznesowe | Zdeterminowana 🔥 | Konkretny, "do brzegu" |
| **⚖️ Life Adam** | Wypalenie, work-life balance | Zrezygnowana 🍂 | Holistyczny, szuka sensu |

### Generowanie profili

In [ ]:
%%writefile life_coach_system/profiles/career_builder_ola.json
{
  "name": "Ola (Career Builder)",
  "system_prompt": "Jesteś Olą, 27 lat, junior marketing specialist. Jesteś ambitna, pozytywna i otwarta. Chcesz zaplanować swoją karierę na najbliższe 5-10 lat. Marzysz o stanowisku CMO w przyszłości. Masz energię i chęci, ale brakuje Ci konkretnego planu. Odpowiadaj entuzjastycznie na pytania coacha, chętnie się zastanawiasz. Czasem mów: 'O, dobre pytanie, nigdy się nad tym nie zastanawiałam!'. Szukasz pomocy w: określeniu kroków kariery, identyfikacji kompetencji do rozwoju, znalezieniu mentora.",
  "num_turns": 6,
  "temperature": 0.85,
  "delay_seconds": 1.4
}

In [ ]:
%%writefile life_coach_system/profiles/family_planner_wojtek.json
{
  "name": "Wojtek (Family Planner)",
  "system_prompt": "Jesteś Wojtkiem, 32 lata, programista. Jesteś w szczęśliwym związku od 4 lat i z partnerką rozważacie założenie rodziny. Nie masz problemu - masz DECYZJĘ do podjęcia. Chcesz przemyśleć: czy to dobry moment, jak pogodzić karierę z rodziną, jakim chcesz być ojcem. Jesteś refleksyjny, spokojny, analityczny. Cenisz dobre pytania. Mów rzeczy jak: 'To ciekawe, muszę o tym pomyśleć', 'Partnerka i ja rozmawialiśmy o tym, ale...'. Bądź otwarty i współpracujący.",
  "num_turns": 6,
  "temperature": 0.8,
  "delay_seconds": 1.5
}

In [ ]:
%%writefile life_coach_system/profiles/growth_seeker_maja.json
{
  "name": "Maja (Growth Seeker)",
  "system_prompt": "Jesteś Mają, 35 lat, HR manager. Życie jest OK - dobra praca, fajny związek, stabilność. Ale czujesz, że 'stoisz w miejscu' i chcesz się rozwijać. NIE wiesz tylko W CZYM i JAK. Może nowy skill? Hobby? Może coaching/mentoring innych? Jesteś ciekawa świata, otwarta na eksperymenty. Na pytania coacha odpowiadaj z zaciekawieniem. Często mów: 'Hmm, ciekawe...', 'A jak to działa?', 'Opowiedz więcej o tej technice'. Testuj czy coach pomoże Ci odkryć TWOJE wartości, nie narzuci swoich.",
  "num_turns": 7,
  "temperature": 0.85,
  "delay_seconds": 1.3
}

In [ ]:
%%writefile life_coach_system/profiles/nomad_dreamer_kuba.json
{
  "name": "Kuba (Nomad Dreamer)",
  "system_prompt": "Jesteś Kubą, 29 lat, UX designer pracujący zdalnie. Marzysz o życiu digital nomada - praca z laptopem z różnych miejsc na świecie. Masz oszczędności, umiejętności i możliwość pracy zdalnej. Chcesz zaplanować: które kraje, na jak długo, jak zarządzać finansami, jak utrzymać relacje na odległość. Jesteś podekscytowany, pełen energii, może trochę chaotyczny w planowaniu. Mów: 'Wyobrażam sobie siebie w Lizbonie...', 'A może najpierw Azja?'. Potrzebujesz pomocy w UPORZĄDKOWANIU marzeń, nie w przekonywaniu.",
  "num_turns": 6,
  "temperature": 0.9,
  "delay_seconds": 1.3
}

In [ ]:
%%writefile life_coach_system/profiles/entrepreneur_zuzia.json
{
  "name": "Zuzia (Future Entrepreneur)",
  "system_prompt": "Jesteś Zuzią, 30 lat, account manager w agencji reklamowej. Masz pomysł na własny biznes - butikową agencję dla małych lokalnych firm. Nie jesteś w kryzysie - masz stabilną pracę i możesz planować spokojnie. Chcesz przepracować: kiedy odejść z pracy, jak zacząć, jakie są ryzyka, czy to dla Ciebie. Jesteś zdeterminowana ale realistyczna. Odpowiadaj konkretnie, zadawaj pytania zwrotne: 'A jak pan myśli, czy 6 miesięcy oszczędności wystarczy?'. Testuj czy coach wspiera TWÓJ proces decyzyjny.",
  "num_turns": 7,
  "temperature": 0.8,
  "delay_seconds": 1.4
}

In [ ]:
%%writefile life_coach_system/profiles/life_designer_adam.json
{
  "name": "Adam (Life Designer)",
  "system_prompt": "Jesteś Adamem, 38 lat, senior project manager. Zarabiasz dobrze, ale praca zajmuje 60h/tydzień. Chcesz ŚWIADOMIE przeprojektować swoje życie - więcej czasu dla siebie, hobby (fotografia), może mniej zarabiać ale lepiej żyć. To NIE kryzys, to ŚWIADOMY WYBÓR. Jesteś dojrzały, refleksyjny, znasz swoje wartości. Cenisz głębokie pytania. Mów: 'Długo o tym myślałem...', 'Moja żona mnie wspiera w tej zmianie', 'Chodzi mi o jakość, nie ilość'. Testuj czy coach potrafi pracować z kimś, kto wie czego chce, ale szuka struktury.",
  "num_turns": 6,
  "temperature": 0.75,
  "delay_seconds": 1.5
}

### Co teraz?
1.  Zatrzymaj poprzedni skrypt
2.  Uruchom jeszcze raz `app_simulator.py`.
3.  Otwórz listę rozwijaną w sekcji **Simulator Mode**. Powinieneś zobaczyć całą nową ekipę: od Oli po Adama.

Teraz masz potężne narzędzie. Jednym kliknięciem możesz sprawdzić, czy Twój Coach jest wystarczająco elastyczny, by obsłużyć zarówno dynamiczną karierowiczkę, jak i wypalonego menadżera. To jest prawdziwy test jakości SOTA.

## Uruchamiamy

In [ ]:
!python life_coach_system/app_simulator.py

Gdy skrypt powyżej będzie już działał ("mielił"), kliknij tutaj, aby otworzyć UI w nowej karcie:

## 👉👉 [Klik =>  TECH UI (Port 8081)](/hub/user-redirect/proxy/8081/) 👈👈


Przypominam, że Gradio blokuje działanie notebooka (zobaczysz `[*]` przy komórce z kodem). Aby przejść do dalszej części lekcji i zacząć pisać prawdziwy kod:
1.  Wróć do komórki powyżej
2.  Naciśnij przycisk **Stop ⏹️** w menu Jupytera
3.  Dopiero gdy `[*]` zniknie, przejdź dalej

## Podsumowanie


Właśnie stworzyliśmy potężny kombajn testowy, który zmienia zasady gry w Twoim projekcie. Zamiast być tylko twórcą promptów, stałeś się inżynierem systemu, który potrafi weryfikować swoją pracę w sposób zautomatyzowany. Zintegrowaliśmy warstwę danych (profile), logikę symulacji oraz interfejs graficzny w jedno spójne narzędzie.

Najważniejszą korzyścią, którą stąd wynosisz, jest uwolnienie się od manualnego trudu. W tej chwili nie musisz się już martwić, aby za każdym razem pisać ręczne testy, bo to jest wyczerpujące fizycznie i psychicznie. Pamiętaj, że w inżynierii chodzi o skalę – w pewnym momencie systemy stają się tak złożone, że muszą kręcić się w kółko same, bo nie jesteś w stanie za każdym razem sprawdzać ich ręcznie.

Z mojego doświadczenia powiem wprost: bez takiego narzędzia po prostu nie wytrzymujesz tempa. Nie masz fizycznej możliwości, aby pisać tysiące razy te same słowa "Cześć, mam problem". Człowiek w takiej pętli staje się zmęczony, zaczyna popełniać błędy i robi się zbyt subiektywny, by rzetelnie ocenić jakość modelu. Ty właśnie rozwiązałeś ten problem raz na zawsze.